# Week 5, Day 2: Matrix multiplication: composing transformations and layers

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w005/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Matrix multiplication from scratch

Write `my_matmul(A, B)` that raises `ValueError` if the inner dimensions do not match, and otherwise builds the product entry by entry as row-times-column dot products.

In [ ]:
import numpy as np

def my_matmul(A, B):
    m, k = A.shape
    k2, n = B.shape
    ...
    C = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            C[i, j] = ...
    return C

A = np.array([[1.0, 2.0], [3.0, 4.0]]); B = np.array([[0.0, 1.0], [1.0, 0.0]])
print(my_matmul(A, B)); print(A @ B)

In [ ]:
import numpy as np
def test_matmul():
    "my_matmul matches A @ B"
    rng = np.random.default_rng(0)
    A, B = rng.normal(size=(3, 4)), rng.normal(size=(4, 2))
    assert np.allclose(my_matmul(A, B), A @ B)

def test_shape_error():
    "my_matmul refuses incompatible shapes"
    try:
        my_matmul(np.ones((2, 3)), np.ones((2, 3)))
    except ValueError:
        return
    assert False, "Raise ValueError when the inner dimensions differ."

test_matmul()
test_shape_error()
print("All checks passed")

### Exercise: Two ways to multiply a data matrix by itself

`X` has 40 trials by 6 neurons. Compute `trial_sim` (every pair of trials' dot product) and `cov` (the covariance between every pair of neurons): centre each neuron's column, compute $X_c^\top X_c$ and divide by the number of trials minus one. Check against `np.cov`.

In [ ]:
rng = np.random.default_rng(3)
shared = rng.normal(size=(40, 1))
X = shared @ np.array([[1.0, 0.8, 0.6, 0.0, 0.0, -0.5]]) + rng.normal(0, 0.5, (40, 6))

trial_sim = ...
Xc = ...
cov = ...
print(np.round(cov, 2))

In [ ]:
import numpy as np
def test_trials():
    "trial_sim holds the dot product of every pair of trials"
    assert trial_sim.shape == (X.shape[0], X.shape[0])
    assert abs(trial_sim[3, 7] - X[3] @ X[7]) < 1e-9

def test_neurons():
    "cov is the neuron-by-neuron covariance matrix"
    assert cov.shape == (X.shape[1], X.shape[1])
    assert np.allclose(cov, np.cov(X, rowvar=False))

test_trials()
test_neurons()
print("All checks passed")

### Exercise: Linear layers collapse; nonlinear ones do not

Compute `W_single`, the one matrix equal to layer `W1` (100 to 50) followed by `W2` (50 to 10), and check it on the inputs `X`. Then insert a ReLU between the layers: fit the best single linear map from `X` to those outputs with `np.linalg.lstsq` (tomorrow's tool), and store the leftover mean squared error in `gap`.

In [ ]:
rng = np.random.default_rng(4)
W1 = rng.normal(size=(50, 100)) / 10
W2 = rng.normal(size=(10, 50)) / 7
X = rng.normal(size=(500, 100))                 # 500 inputs, one per row

W_single = ...
print("linear stack reproduced:", np.allclose(X @ W_single.T, (X @ W1.T) @ W2.T))

Y = np.maximum(0, X @ W1.T) @ W2.T              # with a ReLU between the layers
B, *_ = np.linalg.lstsq(X, Y, rcond=None)       # best linear map, shape (100, 10)
gap = ...
print("error of the best single linear layer:", round(gap, 4))

In [ ]:
import numpy as np
def test_collapse():
    "W_single reproduces the two linear layers exactly"
    assert W_single.shape == (10, 100)
    assert np.allclose(X @ W_single.T, (X @ W1.T) @ W2.T)

def test_nonlinear():
    "with a ReLU in between, no single matrix fits exactly"
    assert gap > 1e-3, f"gap was {gap}"

test_collapse()
test_nonlinear()
print("All checks passed")